# Long Steps in Gradient Descent Methods

## Install

In [ ]:
# Quick environment check
import sys, platform
print(f"Python: {sys.version.split()[0]}, Platform: {platform.system()}-{platform.release()}")

## Motivation: When Newton Fails

We're used to Newton—when it works, the convergence is superb. But it's not always a good idea:

1. Newton–Kantorovich requires local Lipschitz differentiability of the residual and invertibility at the minimizer; easy to violate.
2. At very high order, nonlinearity makes assembly costly.
3. For nonlinear problems with many solutions, initial guesses may lie outside any basin of attraction.

We'll see a simple non-smooth energy where Newton breaks down, and why gradient descent is more robust.

## Back to Basics: Gradient Descent

For energy minimization, gradient descent (GD) updates

- x_{k+1} = x_k - \alpha \nabla E(x_k)

Key facts:
- Converges for fixed step sizes in (0, 2/L), with L the Lipschitz constant of \nabla E.
- Only requires one lower level of regularity than Newton.
- The linear operator to invert in function spaces is the inner product; often identical across steps.
- Global convergence to a minimizer for convex energies regardless of start.

## Set Up: Imports and Plotting Utilities

We'll import NumPy, SciPy, Matplotlib; set seeds; and define small helpers for plotting trajectories and energy vs iterations.

In [ ]:
# Imports and plotting helpers
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm
np.random.seed(42)
plt.rcParams.update({"figure.figsize": (6,4), "axes.grid": True})

# Optional SciPy for eigen computations if available
try:
    import scipy.sparse.linalg as spla
except Exception:
    spla = None

def plot_energy(iter_energies, labels=None, title="Energy vs Iteration"):
    plt.figure()
    if labels is None:
        labels = [None]*len(iter_energies)
    for E_seq, lbl in zip(iter_energies, labels):
        plt.plot(E_seq, label=lbl)
    plt.xlabel("Iteration k")
    plt.ylabel("E(x_k)")
    if any(lbl is not None for lbl in labels):
        plt.legend()
    plt.title(title)
    plt.show()

def plot_trajectory(trajs, labels=None, title="Trajectory in 2D"):
    plt.figure()
    if labels is None:
        labels = [None]*len(trajs)
    for T, lbl in zip(trajs, labels):
        T = np.array(T)
        plt.plot(T[:,0], T[:,1], marker="o", label=lbl)
    plt.xlabel("x")
    plt.ylabel("y")
    if any(lbl is not None for lbl in labels):
        plt.legend()
    plt.title(title)
    plt.axis("equal")
    plt.show()

def savefig(path):
    plt.savefig(path, bbox_inches='tight')
    print(f"Saved figure to {path}")

def log(msg):
    print(msg)

## Helper API: Energy, Gradient, Lipschitz Estimation

We'll provide callables for energies, gradients, and an estimator for the Lipschitz constant L of ∇E on a bounded domain.

In [ ]:
# Energy/gradient helpers and L-estimation
import numpy as np
from numpy.linalg import eigvals
from numpy.random import default_rng
rng = default_rng(123)

def estimate_L_quadratic(A):
    # For quadratic E(x)=0.5 x^T A x + b^T x with SPD A, L = lambda_max(A)
    return float(np.max(np.real(eigvals(A))))

def grad_quadratic(A, b, x):
    return A @ x + b

def energy_quadratic(A, b, x):
    return 0.5 * x.T @ A @ x + b.T @ x

def estimate_L_box_hessian(hess_fun, bounds, samples=256):
    # Sample Hessians in a box and estimate spectral norm upper bound
    lows, highs = np.array(bounds[0]), np.array(bounds[1])
    max_eval = 0.0
    for _ in range(samples):
        xi = rng.uniform(lows, highs)
        H = hess_fun(xi)
        # symmetric approx
        evals = np.linalg.eigvalsh((H + H.T)/2)
        max_eval = max(max_eval, float(np.max(np.abs(evals))))
    return max_eval

## Newton vs GD on a Non-smooth Toy Energy

Consider the 2D energy E(x, y) = |x| + 0.5 y^2.
- GD uses a subgradient for |x|: ∂|x| ∈ [-1,1] at x=0; we'll take 0 for stability.
- Newton fails: the Hessian of |x| is undefined at x=0 and 0 elsewhere, leading to non-invertibility.
We'll plot trajectories to illustrate GD robustness and Newton breakdown.

In [ ]:
# Non-smooth toy energy: GD vs Newton attempt
def energy_abs(x):
    # x is (2,) with (x,y)
    return np.abs(x[0]) + 0.5*(x[1]**2)

def grad_abs(x):
    gx = 0.0 if np.isclose(x[0], 0.0) else (1.0 if x[0] > 0 else -1.0)
    gy = x[1]
    return np.array([gx, gy])

def hess_abs(x):
    # Hessian of |x| is undefined at 0 and 0 elsewhere; we return zeros
    return np.zeros((2,2))

def gd_run_abs(x0, alpha=0.5, iters=40):
    xs = [x0.copy()]
    E = [energy_abs(x0)]
    x = x0.copy()
    for k in range(iters):
        g = grad_abs(x)
        x = x - alpha*g
        xs.append(x.copy())
        E.append(energy_abs(x))
    return np.array(xs), np.array(E)

def newton_attempt_abs(x0, iters=10):
    # Formal Newton: x_{k+1} = x_k - H^{-1} g; but H is zero, non-invertible
    xs = [x0.copy()]
    x = x0.copy()
    for k in range(iters):
        H = hess_abs(x)
        g = grad_abs(x)
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            # Fallback: pseudo-inverse or break
            # Using pseudo-inverse shows instability/large steps
            step = np.linalg.pinv(H) @ g
        x = x - step
        xs.append(x.copy())
    return np.array(xs)

x0 = np.array([1.0, 1.5])
traj_gd, E_gd = gd_run_abs(x0, alpha=0.5, iters=40)
traj_newton = newton_attempt_abs(x0, iters=8)
plot_trajectory([traj_gd, traj_newton], labels=["GD", "Newton (attempt)"], title="Non-smooth energy: GD vs Newton attempt")
plot_energy([E_gd], labels=["GD"], title="Energy decay: Non-smooth energy")

## GD on 2D Quadratic: Step Sizes in (0, 2/L)

Define E(x) = 0.5 x^T A x + b^T x with SPD A. Compute L = λ_max(A). Test fixed step sizes to show convergence inside (0, 2/L) and divergence just outside.

In [ ]:
# Quadratic GD demos: fixed step sizes
import numpy as np
A = np.array([[5.0, 0.0],[0.0, 1.0]])  # ill-conditioned-ish
b = np.array([-1.0, 2.0])
x_star = -np.linalg.solve(A, b)
L = estimate_L_quadratic(A)
alphas = [0.25/L, 0.9/L, 1.5/L, 2.1/L]
labels = [f"alpha={a:.3g}" for a in alphas]

def gd_run_quadratic(x0, alpha, iters=50):
    xs = [x0.copy()]
    Es = [energy_quadratic(A, b, x0)]
    x = x0.copy()
    for k in range(iters):
        g = grad_quadratic(A, b, x)
        x = x - alpha*g
        xs.append(x.copy())
        Es.append(energy_quadratic(A, b, x))
    return np.array(xs), np.array(Es)

x0 = np.array([2.0, -2.0])
traj_list = []
E_list = []
for a in alphas:
    traj, Es = gd_run_quadratic(x0, a)
    traj_list.append(traj)
    E_list.append(Es)
plot_trajectory(traj_list, labels=labels, title=f"Quadratic GD trajectories (L={L:.3g})")
plot_energy(E_list, labels=labels, title="Quadratic GD: energy vs iteration")
log(f"Strong convexity L={L:.3g}; bound 2/L={(2/L):.3g}")

## Steepest Descent with Exact Line Search (Quadratic)

For quadratic energies, exact line search for steepest descent yields a closed form:

α_k = (∇E(x_k)^T ∇E(x_k)) / (∇E(x_k)^T A ∇E(x_k)).

We'll run iterations and log α_k, E(x_k), and residual norms.

In [ ]:
# Steepest descent with exact line search on quadratic
def steepest_descent_exact(A, b, x0, iters=50):
    xs = [x0.copy()]
    Es = [energy_quadratic(A, b, x0)]
    alphas = []
    residuals = [norm(grad_quadratic(A, b, x0))]
    x = x0.copy()
    for k in range(iters):
        g = grad_quadratic(A, b, x)
        denom = float(g.T @ A @ g)
        alpha_k = float(g.T @ g) / denom if denom > 0 else 0.0
        x = x - alpha_k*g
        xs.append(x.copy())
        Es.append(energy_quadratic(A, b, x))
        residuals.append(norm(grad_quadratic(A, b, x)))
        alphas.append(alpha_k)
    return np.array(xs), np.array(Es), np.array(alphas), np.array(residuals)

x0 = np.array([2.0, -2.0])
traj_ls, E_ls, alphas_ls, res_ls = steepest_descent_exact(A, b, x0)
plot_energy([E_ls], labels=["Exact line search"], title="Energy decay: exact line search (quadratic)")
plt.figure()
plt.plot(alphas_ls, marker='o')
plt.axhline(2.0/L, color='r', linestyle='--', label='2/L bound')
plt.xlabel('Iteration k')
plt.ylabel('alpha_k')
plt.title('Line search step sizes')
plt.legend()
plt.show()

## Observe Alternating Step Sizes from Line Search

For ill-conditioned A, α_k often alternates between small and large values; some α_k may exceed 2/L yet still decrease energy due to exact line search.

## Periodic Step-Size Patterns and Acceleration

Extract representative small/large step sizes from line search and apply them periodically:

α_k = α_small if k even, else α_large.

Compare convergence vs fixed α and exact line search.

In [ ]:
# Periodic alternating step sizes inspired by line search
def gd_periodic(A, b, x0, alpha_small, alpha_large, iters=50):
    xs = [x0.copy()]
    Es = [energy_quadratic(A, b, x0)]
    x = x0.copy()
    for k in range(iters):
        g = grad_quadratic(A, b, x)
        alpha_k = alpha_small if (k % 2 == 0) else alpha_large
        x = x - alpha_k*g
        xs.append(x.copy())
        Es.append(energy_quadratic(A, b, x))
    return np.array(xs), np.array(Es)

alpha_small = float(np.percentile(alphas_ls, 25)) if len(alphas_ls) else 0.5/L
alpha_large = float(np.percentile(alphas_ls, 75)) if len(alphas_ls) else 1.5/L
traj_per, E_per = gd_periodic(A, b, x0, alpha_small, alpha_large)
plot_energy([E_list[1], E_ls, E_per], labels=[labels[1], "Line search", f"Periodic ({alpha_small:.3g},{alpha_large:.3g})"], title="Periodic vs fixed vs line search")

## GD on Non-Strongly-Convex Energy E(x,y)=x^4+y^2

We'll visualize the O(1/k) energy convergence trend for a quartic + quadratic energy, contrasting with the faster decay in the strongly convex quadratic case.

In [ ]:
# Non-strongly-convex energy: x^4 + y^2
def energy_quartic(x):
    return x[0]**4 + x[1]**2

def grad_quartic(x):
    return np.array([4*x[0]**3, 2*x[1]])

def hess_quartic(x):
    return np.array([[12*x[0]**2, 0.0],[0.0, 2.0]])

def gd_run_quartic(x0, alpha_func, iters=200):
    xs = [x0.copy()]
    Es = [energy_quartic(x0)]
    x = x0.copy()
    for k in range(iters):
        g = grad_quartic(x)
        alpha = alpha_func(x)
        x = x - alpha*g
        xs.append(x.copy())
        Es.append(energy_quartic(x))
    return np.array(xs), np.array(Es)

bounds = (np.array([-2.0,-2.0]), np.array([2.0,2.0]))
L_local = estimate_L_box_hessian(hess_quartic, bounds)
def alpha_local(x):
    return 1.0 / L_local

x0q = np.array([1.0, 1.0])
traj_q, E_q = gd_run_quartic(x0q, alpha_local)
plot_energy([E_q], labels=["Quartic GD (alpha=1/L_local)"])
plt.figure()
plt.loglog(np.arange(len(E_q)), E_q - np.min(E_q), marker='o')
plt.loglog(np.arange(1, len(E_q)), 1.0/np.arange(1, len(E_q)), linestyle='--', label='C/k reference')
plt.xlabel('Iteration k')
plt.ylabel('E(x_k)-E(x*)')
plt.title('Quartic energy: O(1/k) trend (heuristic)')
plt.legend()
plt.show()

## Convergence Metrics: ||x_k−x*|| vs E(x_k)−E(x*)

We'll compute and compare both norms and energy gaps for quadratic and quartic examples. The energy gap is often more robust across problems.

In [ ]:
# Metrics comparison
def compute_metrics_quadratic(traj):
    xs = np.array(traj)
    Es = [energy_quadratic(A, b, x) for x in xs]
    E_star = energy_quadratic(A, b, x_star)
    norms = [norm(x - x_star) for x in xs]
    return np.array(Es), E_star, np.array(norms)

Es_fixed, E_star_q, norms_fixed = compute_metrics_quadratic(traj_list[1])  # using alpha=0.9/L
Es_ls, _, norms_ls = compute_metrics_quadratic(traj_ls)

plt.figure()
plt.semilogy(Es_fixed - E_star_q, label='Fixed alpha energy gap')
plt.semilogy(Es_ls - E_star_q, label='Line search energy gap')
plt.xlabel('Iteration k')
plt.ylabel('E(x_k)-E(x*)')
plt.title('Quadratic: energy gap convergence')
plt.legend()
plt.show()

plt.figure()
plt.semilogy(norms_fixed, label='Fixed alpha ||x_k-x*||')
plt.semilogy(norms_ls, label='Line search ||x_k-x*||')
plt.xlabel('Iteration k')
plt.ylabel('||x_k-x*||')
plt.title('Quadratic: state error convergence')
plt.legend()
plt.show()

## Finite-Element-Flavored Step: Mass-Matrix Inner Product in 1D

Assemble a simple 1D mesh with mass matrix M and stiffness K.
Define E(u)=0.5 u^T K u − f^T u and perform GD in the M-inner product:

Solve M s_k = ∇E(u_k) then u_{k+1}=u_k−α s_k.

The linear operator (M) is identical each step; we can pre-factorize.

In [ ]:
# 1D FE-like mass and stiffness matrices and M-inner product GD
def assemble_1d(n):
    # Uniform mesh [0,1] with n+1 nodes, h=1/n, linear elements
    h = 1.0/n
    # Mass matrix (lumped or consistent). We'll do consistent mass.
    M = np.zeros((n+1, n+1))
    K = np.zeros((n+1, n+1))
    for i in range(n):
        # local nodes i, i+1
        # Consistent mass for linear element length h: (h/6)[[2,1],[1,2]]
        M[i,i]     += 2*h/6
        M[i,i+1]   += 1*h/6
        M[i+1,i]   += 1*h/6
        M[i+1,i+1] += 2*h/6
        # Stiffness: (1/h)[[1,-1],[-1,1]]
        K[i,i]     += 1/h
        K[i,i+1]   += -1/h
        K[i+1,i]   += -1/h
        K[i+1,i+1] += 1/h
    return M, K

def gd_mass_inner_product(K, f, M, u0, alpha, iters=50):
    from numpy.linalg import solve
    us = [u0.copy()]
    Es = [0.5*u0.T@K@u0 - f.T@u0]
    u = u0.copy()
    # Pre-factorize M via LU (solve uses factorization internally)
    for k in range(iters):
        grad = K@u - f
        s = solve(M, grad)
        u = u - alpha*s
        us.append(u.copy())
        Es.append(0.5*u.T@K@u - f.T@u)
    return np.array(us), np.array(Es)

n = 50
M, K = assemble_1d(n)
# Force term f from a sine load, zero BCs implicitly (no modification for simplicity)
xcoords = np.linspace(0,1,n+1)
f = np.sin(2*np.pi*xcoords)
u0 = np.zeros(n+1)
# Estimate L via largest eigenvalue of M^{-1}K in M-inner product
# For simplicity, approximate L by largest eigenvalue of K wrt Euclidean norm
L_fe = float(np.max(np.linalg.eigvalsh(K)))
alpha_fe = 0.9/L_fe
traj_fe, E_fe = gd_mass_inner_product(K, f, M, u0, alpha_fe, iters=80)
plot_energy([E_fe], labels=["M-inner-product GD"], title="FE-flavored GD: energy vs iteration")

## 2D Demo Scripts from Repository

We'll optionally run your existing demo scripts for 2D gradient descent, line search, and periodic steps. If they rely on a main guard, we can import and call functions; otherwise, we can exec their contents in-process.

In [ ]:
# Safe runner for repository Python scripts inside the notebook
import os, runpy, sys
repo_root = r"c:\\Users\\boris\\OneDrive\\Documents\\My Stuff\\Repositories\\Mine\\Other\\07. Long Steps"
def run_script(rel_path):
    path = os.path.join(repo_root, rel_path)
    if not os.path.exists(path):
        print(f"Script not found: {path}")
        return
    print(f"Running {rel_path}...")
    # Avoid polluting global namespace; run in its own dict
    try:
        runpy.run_path(path, run_name='__main__')
    except SystemExit as e:
        print(f"Script exited: {e}")
    except Exception as e:
        print(f"Error running {rel_path}: {e}")

# Uncomment to run your demos if desired during the talk
# run_script(os.path.join('Code','2D','gradient_descent_demo.py'))
# run_script(os.path.join('Code','2D','gradient_descent_linesearch_demo.py'))
# run_script(os.path.join('Code','2D','gradient_descent_periodic_demo.py'))